Importing Libraries

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.losses import MeanSquaredError

Import Data Setup

In [2]:
y_true = [3, 5, 2, 7]
y_pred = [2.5, 5, 4, 8]

print("Actual values (y_true):", y_true)
print("Predicted values (y_pred):", y_pred)

Actual values (y_true): [3, 5, 2, 7]
Predicted values (y_pred): [2.5, 5, 4, 8]


MSE using Plain Python (For Loop, No NumPy)

In [3]:
def mse_plain_python(y_true, y_pred):
    n = len(y_true)
    total_squared_error = 0.0
    for actual, pred in zip(y_true, y_pred):
        squared_error = (actual - pred) ** 2
        total_squared_error += squared_error
    return total_squared_error / n

# Calculate MSE
mse_python = mse_plain_python(y_true, y_pred)
print(f"MSE (Plain Python): {mse_python}")

MSE (Plain Python): 1.3125


MSE using NumPy (Vectorized, No For Loop)

In [4]:
def mse_numpy(y_true, y_pred):
    actual = np.array(y_true)
    pred = np.array(y_pred)
    return np.mean((actual - pred) ** 2)

# Calculate MSE
mse_np = mse_numpy(y_true, y_pred)
print(f"MSE (NumPy): {mse_np}")

MSE (NumPy): 1.3125


MSE using Keras Loss (MeanSquaredError)

In [5]:
def mse_keras(y_true, y_pred):
    mse_loss = MeanSquaredError()
    return mse_loss(y_true, y_pred).numpy()

# Calculate MSE
mse_tf = mse_keras(y_true, y_pred)
print(f"MSE (Keras): {mse_tf}")

MSE (Keras): 1.3125


Final Results

In [7]:
print(f"MSE (Plain Python) : {mse_python}")
print(f"MSE (NumPy)        : {mse_np}")
print(f"MSE (Keras)        : {mse_tf}")




MSE (Plain Python) : 1.3125
MSE (NumPy)        : 1.3125
MSE (Keras)        : 1.3125


Importing Libraries

In [8]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical


np.random.seed(42)
tf.random.set_seed(42)

Load and Preprocess MNIST Dataset

In [10]:
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

# 2. Flatten images (28x28 = 784) and scale pixels to range [0, 1]
X_train = X_train_raw.reshape(-1, 784).astype('float32') / 255.0
X_test = X_test_raw.reshape(-1, 784).astype('float32') / 255.0

# 3. Convert target labels to One-Hot Encoding
y_train = to_categorical(y_train_raw, num_classes=10)
y_test = to_categorical(y_test_raw, num_classes=10)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)

X_train shape: (60000, 784)
X_test shape : (10000, 784)
y_train shape: (60000, 10)


Model Builder and Training Pipeline Function

In [13]:
def experiment_loss_function(loss_name):

    tf.random.set_seed(42)


    model = Sequential([
        Input(shape=(784,)),
        Dense(32, activation='relu'),
        Dense(10, activation='softmax')
    ])


    model.compile(
        optimizer='adam',
        loss=loss_name,
        metrics=['accuracy']
    )
    print(f"  TRAINING MODEL WITH LOSS: {loss_name}")

    # Train for 5 epochs
    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=32,
        verbose=1
    )


    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n[RESULT] Test Accuracy ({loss_name}): {test_acc * 100:.2f}%")

    return test_acc

Run Training Across All Three Loss Functions

In [14]:
loss_experiments = [
    'categorical_crossentropy',
    'mean_squared_error',
    'mean_absolute_error'
]


accuracy_results = {}


for loss_fn in loss_experiments:
    acc = experiment_loss_function(loss_fn)
    accuracy_results[loss_fn] = acc


  TRAINING MODEL WITH LOSS: categorical_crossentropy
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8979 - loss: 0.3608
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9437 - loss: 0.1964
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9558 - loss: 0.1526
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9631 - loss: 0.1271
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9675 - loss: 0.1097

[RESULT] Test Accuracy (categorical_crossentropy): 96.11%

  TRAINING MODEL WITH LOSS: mean_squared_error
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8980 - loss: 0.0158
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9421 - loss: 0.0090
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9531 - loss: 0.0075
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9598 - loss: 0.0064
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - a

Comparative Summary Table and Final Analysis

In [15]:
summary_df = pd.DataFrame({
    'Loss Function': list(accuracy_results.keys()),
    'Test Accuracy': [f"{val * 100:.2f}%" for val in accuracy_results.values()]
})
print("           ACCURACY COMPARISON TABLE              ")
print(summary_df.to_string(index=False))
print("--- Analysis of Loss Functions ---")


           ACCURACY COMPARISON TABLE              
           Loss Function Test Accuracy
categorical_crossentropy        96.11%
      mean_squared_error        95.89%
     mean_absolute_error        86.00%
--- Analysis of Loss Functions ---
1. Categorical Crossentropy worked best because its logarithmic penalty optimizes classification probabilities aggressively against soft targets.
2. Mean Squared Error (MSE) performed moderately well but converges slower due to vanishing gradient regions inherent in quadratic loss on Softmax outputs.
3. Mean Absolute Error (MAE) worked worst because its fixed gradient scale causes unstable weight adjustments around small prediction error bounds.
